# ColQwen validation: encode ViDoRe on a Kaggle GPU

Encodes the two ViDoRe V1 splits used for ColPali (`docvqa_test_subsampled`,
`infovqa_test_subsampled`, 500 pages each) with **ColQwen2** and **ColQwen2.5**,
and writes OptiVision vector files. Nothing is measured here: every compression
and calibration study runs afterwards, on CPU, with the same scripts as ColPali.

**Before running**
1. *Settings -> Accelerator*: **GPU T4 x2** (prefer T4 over P100: recent torch
   builds may lack P100 kernels). Only one GPU is used.
2. *Settings -> Internet*: **On** (needs a phone-verified Kaggle account).
3. *Save Version -> Save & Run All (Commit)*, so the run survives closing the
   tab. About 1-2 hours. Then download `colqwen_vectors.zip` from the *Output* tab.

Each step stops the notebook if a check fails, so a bad run cannot produce
plausible-looking vectors.

In [ ]:
import subprocess, sys, json, os
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
import torch
assert torch.cuda.is_available(), "No GPU: set Settings -> Accelerator -> GPU T4 x2"
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0), "| capability", torch.cuda.get_device_capability(0))

## 1. Install

The checkout is deliberately **not** named `optivision`: on Kaggle the working
directory is on `sys.path`, and a folder with the package's name would shadow it.

In [ ]:
BRANCH = "main"  # R11 ran at commit 46adfd9; check it out to reproduce exactly
WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/optivision-rag"
OUT = f"{WORK}/vectors"
!rm -rf {REPO_DIR} && git clone -q --depth 1 --branch {BRANCH} https://github.com/Daemon-VI/optivision-rag.git {REPO_DIR}
!cd {REPO_DIR} && git log --oneline -1
!pip install -q -e "{REPO_DIR}[bench,merge]" "colpali-engine==0.3.17" 2>&1 | tail -3
# Kaggle preinstalls torchao 0.10; peft 0.19 refuses to apply a LoRA adapter while an
# incompatible torchao is importable. Nothing here uses torchao, so remove it.
!pip uninstall -y -q torchao
!python -c "import torch, transformers, colpali_engine, peft, importlib.util as u; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| transformers', transformers.__version__, '| peft', peft.__version__, '| torchao present:', u.find_spec('torchao') is not None)"
os.makedirs(OUT, exist_ok=True)

## 2. Smoke test ColQwen2, two ways

The same model is loaded as the authors' pre-merged checkpoint and as the
adapter-only repo merged by our loader. If the two agree (row-wise cosine above
0.99), the adapter path that ColQwen2.5 depends on is trustworthy.

In [ ]:
r = subprocess.run([sys.executable, f"{REPO_DIR}/scripts/colqwen_smoke.py",
                    "--model", "vidore/colqwen2-v1.0-merged", "--backend", "colqwen2",
                    "--compare-model", "vidore/colqwen2-v1.0", "--pages", "6",
                    "--out", f"{OUT}/smoke_colqwen2.json"], capture_output=True, text=True)
print(r.stdout[-4000:]); print(r.stderr[-2000:])
assert r.returncode == 0, "ColQwen2 smoke test failed: do not encode, inspect the report above"

## 3. Encode ColQwen2 (both splits)

In [ ]:
def encode(model, backend, dataset, extra=()):
    r = subprocess.run([sys.executable, f"{REPO_DIR}/scripts/encode_vectors.py", model, dataset,
                        "--backend", backend, "--out", OUT, *extra], capture_output=True, text=True)
    print(r.stdout[-3000:]); print(r.stderr[-1500:])
    assert r.returncode == 0, f"encoding {model} on {dataset} failed"

for ds in ("vidore/docvqa_test_subsampled", "vidore/infovqa_test_subsampled"):
    encode("vidore/colqwen2-v1.0-merged", "colqwen2", ds)

## 4. Smoke test ColQwen2.5

No pre-merged release exists, so the loader merges the adapter itself and
refuses to continue unless the merge changed the weights it targets.

In [ ]:
import shutil, glob
# free disk: the ColQwen2 weights are no longer needed
for d in glob.glob(os.path.expanduser("~/.cache/huggingface/hub/models--vidore--colqwen2-*")):
    shutil.rmtree(d, ignore_errors=True)
r = subprocess.run([sys.executable, f"{REPO_DIR}/scripts/colqwen_smoke.py",
                    "--model", "vidore/colqwen2.5-v0.2", "--backend", "colqwen2.5", "--pages", "6", "--multi-gpu",
                    "--out", f"{OUT}/smoke_colqwen2.5.json"], capture_output=True, text=True)
print(r.stdout[-4000:]); print(r.stderr[-2000:])
assert r.returncode == 0, "ColQwen2.5 smoke test failed: do not encode, inspect the report above"

## 5. Encode ColQwen2.5 (both splits)

In [ ]:
for ds in ("vidore/docvqa_test_subsampled", "vidore/infovqa_test_subsampled"):
    # 14 GiB in float32: split over both T4s rather than drop to float16
    encode("vidore/colqwen2.5-v0.2", "colqwen2.5", ds, extra=("--multi-gpu",))

## 6. Package the results

In [ ]:
!pip freeze > {OUT}/pip_freeze.txt
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv > {OUT}/gpu.txt
!cd {REPO_DIR} && git rev-parse HEAD > {OUT}/commit.txt
!ls -la {OUT}
!cd {WORK} && rm -f colqwen_vectors.zip && zip -q -r colqwen_vectors.zip vectors && ls -la colqwen_vectors.zip